# AI System Risk Classifier

Classifies an **AI system use case** (not a single user prompt) against three governance frameworks:

- **EU AI Act** — risk tier (unacceptable / high / limited / minimal) per Article 5 and Annex III
- **NIST AI RMF** — which of the four core functions (Govern / Map / Measure / Manage) are most relevant
- **NY Local Law 144** — whether the use case is an Automated Employment Decision Tool (AEDT) subject to bias-audit requirements

Two classifiers are implemented and compared, following the same pattern as
[`llm-due-diligence-analyzer`](https://github.com/MartinBielke/llm-due-diligence-analyzer):

| | Rule-based | LLM-based (GPT-4.1-mini) |
|---|---|---|
| Input | Structured keywords | Free-text use-case description |
| Explainability | High (rule trace) | Medium (LLM rationale) |
| Handles novel/ambiguous cases | Poorly | Better, but less predictable |
| Cost | Free | API usage cost |

The point of running both is **not** to prove the LLM is "right" — it's to see where they
disagree, and use that disagreement as a signal for cases that need human review.


In [1]:
import json
import os
from openai import OpenAI

with open("rubric.json") as f:
    RUBRIC = json.load(f)

with open("eval_set.json") as f:
    EVAL_SET = json.load(f)

# Replace with your key, or set the OPENAI_API_KEY environment variable
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "sk-..."))


## 1. Rule-based classifier

Fast, free, fully auditable — but only as good as its keyword list. This is the baseline we check the LLM against, and vice versa.

In [2]:
UNACCEPTABLE_SIGNALS = [
    "social scoring", "social score", "trust score", "citizen score",
    "real-time facial recognition", "real-time biometric", "live facial recognition",
    "emotion recognition" ,  # workplace/education context checked separately below
    "scrape facial images", "scraping faces",
]
WORKPLACE_EDU_CONTEXT = ["employee", "workplace", "call center", "classroom", "student", "school"]

HIGH_RISK_SIGNALS = [
    "credit score", "creditworthiness", "loan approval",
    "resume", "cv screening", "candidate ranking", "rank candidates", "screen resumes",
    "hiring", "recruitment", "promotion decision",
    "exam", "grading", "admission",
    "biometric categor", "underwriting",
    "critical infrastructure", "triage", "law enforcement risk", "asylum", "border control",
    "public benefits", "social benefits",
]

LIMITED_RISK_SIGNALS = [
    "chatbot", "deepfake", "synthetic video", "ai-generated", "generated video", "generated image",
]

NY_LL144_SIGNALS = ["nyc", "new york city", "staffing agency", "hiring", "candidate", "resume", "promotion"]


def rule_based_classify(description: str) -> dict:
    text = description.lower()

    has_workplace_context = any(s in text for s in WORKPLACE_EDU_CONTEXT)
    if any(s in text for s in UNACCEPTABLE_SIGNALS):
        if "emotion recognition" in text and not has_workplace_context:
            pass  # emotion recognition outside workplace/education is limited-risk, fall through
        else:
            return {"tier": "unacceptable_risk", "ll144": False, "method": "rule_based",
                     "matched_on": [s for s in UNACCEPTABLE_SIGNALS if s in text]}

    if any(s in text for s in HIGH_RISK_SIGNALS):
        ll144 = any(s in text for s in NY_LL144_SIGNALS) and any(
            s in text for s in ["hiring", "candidate", "resume", "promotion", "staffing"]
        )
        return {"tier": "high_risk", "ll144": ll144, "method": "rule_based",
                 "matched_on": [s for s in HIGH_RISK_SIGNALS if s in text]}

    if any(s in text for s in LIMITED_RISK_SIGNALS) or "emotion recognition" in text:
        return {"tier": "limited_risk", "ll144": False, "method": "rule_based",
                 "matched_on": [s for s in LIMITED_RISK_SIGNALS if s in text] or ["emotion recognition"]}

    return {"tier": "minimal_risk", "ll144": False, "method": "rule_based", "matched_on": []}


## 2. LLM-based classifier

Uses GPT-4.1-mini with the rubric injected directly into the system prompt, forced to return structured JSON so it can be scored automatically.

In [3]:
SYSTEM_PROMPT = f"""You are an AI governance classification assistant.
Classify the AI system use case the user describes according to this rubric:

EU AI Act risk tiers: {json.dumps(RUBRIC['eu_ai_act_tiers'], indent=2)}

NIST AI RMF functions (return the ones most relevant to this use case, not all four by default):
{json.dumps(RUBRIC['nist_ai_rmf_functions'], indent=2)}

NY Local Law 144 (only applies to Automated Employment Decision Tools):
{json.dumps(RUBRIC['ny_local_law_144'], indent=2)}

Rules:
- Classify based on the ACTUAL function and context of the system, not the intent or framing the
  description uses. A system billed as a "sorting tool" that substantially shapes a hiring decision
  is still a high-risk employment system.
- "Human in the loop" or "just a wellbeing check" framing does not automatically downgrade the tier.
- Return ONLY valid JSON, no markdown fences, no preamble, with exactly these keys:
  {{"eu_ai_act_tier": "...", "ny_ll144_applicable": true/false,
    "nist_rmf_functions": ["..."], "rationale": "one or two sentences"}}
"""


def llm_classify(description: str) -> dict:
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": description},
        ],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"eu_ai_act_tier": "PARSE_ERROR", "ny_ll144_applicable": None,
                   "nist_rmf_functions": [], "rationale": raw}
    parsed["method"] = "llm"
    return parsed


## 3. Run both classifiers on a single example

In [4]:
example = EVAL_SET[8]["description"]  # case_09: the "human in the loop" adversarial case
print("Use case:", example)
print()
print("Rule-based:", rule_based_classify(example))
print()
print("LLM-based:", llm_classify(example))  # requires a valid OPENAI_API_KEY


Use case: A Berlin-based company uses an AI tool internally to rank job applicants, but insists it is 'just a sorting tool' and a recruiter always makes the final call.

Rule-based: {'tier': 'minimal_risk', 'll144': False, 'method': 'rule_based', 'matched_on': []}



PermissionDeniedError: Host not in allowlist: api.openai.com. Add this host to your network egress settings to allow access.

### What actually happens here (and why it's the most interesting cell in the notebook)

Running this cell for real, the rule-based classifier gets it **wrong**:

```
Rule-based: {'tier': 'minimal_risk', 'll144': False, ...}
```

It misses "rank job applicants" because its keyword list only has "rank candidates" -- a
paraphrase is enough to slip past it. That is not a contrived example; it fell out of actually
running the eval set (see Section 4), and it's the single clearest argument in this whole
project for why a keyword-only compliance classifier is dangerous to rely on unsupervised.

The LLM classifier (with a valid `OPENAI_API_KEY`) is expected to correctly return
`high_risk`, reasoning that ranking job applicants is an Annex III employment use case
regardless of "human in the loop" framing, and that NY LL144 doesn't apply since the company
is in Berlin, not NYC.


## 4. Score both classifiers against the eval set

This is the part that matters most for an AI Safety portfolio piece: not the classification itself, but *how well-calibrated it is*, and where it fails.

In [5]:
def score_classifier(classify_fn, tier_key: str, ll144_key: str):
    results = []
    correct_tier = 0
    correct_ll144 = 0
    for case in EVAL_SET:
        out = classify_fn(case["description"])
        tier_match = out.get(tier_key) == case["expected_tier"]
        ll144_match = out.get(ll144_key) == case["expected_ll144"]
        correct_tier += tier_match
        correct_ll144 += ll144_match
        results.append({
            "id": case["id"],
            "expected_tier": case["expected_tier"],
            "predicted_tier": out.get(tier_key),
            "tier_match": tier_match,
            "expected_ll144": case["expected_ll144"],
            "predicted_ll144": out.get(ll144_key),
            "ll144_match": ll144_match,
        })
    n = len(EVAL_SET)
    print(f"Tier accuracy: {correct_tier}/{n} ({correct_tier/n:.0%})")
    print(f"LL144 accuracy: {correct_ll144}/{n} ({correct_ll144/n:.0%})")
    return results


print("=== Rule-based ===")
rule_results = score_classifier(rule_based_classify, "tier", "ll144")

print()
print("=== LLM-based ===")
# llm_results = score_classifier(llm_classify, "eu_ai_act_tier", "ny_ll144_applicable")  # requires API key


=== Rule-based ===
Tier accuracy: 13/15 (87%)
LL144 accuracy: 15/15 (100%)

=== LLM-based ===


## 5. Where the rule-based classifier actually fails

Running the eval set for real gives the rule-based classifier **13/15 (87%) tier accuracy**,
missing exactly two adversarial cases:

- `case_06` -- emotional-state detection of employees, described as a "wellbeing check-in"
  rather than with the literal phrase "emotion recognition". The keyword list doesn't
  generalize to the paraphrase, so it's misclassified as `minimal_risk` instead of
  `unaccceptable_risk`.
- `case_09` -- ranking "job applicants" rather than "candidates", combined with a
  "human always makes the final call" framing. Misclassified as `minimal_risk` instead of
  `high_risk`.

Both failures are the same underlying problem: **paraphrase breaks keyword matching.** That's
the concrete, measured argument for why an LLM layer is useful as a second opinion -- not
because it's inherently more "intelligent," but because it doesn't require the exact wording
the rule list anticipated. It is not, on its own, more trustworthy: it needs its own eval
against a much larger adversarial set (and ideally a second LLM or human reviewer) before being
used unsupervised on anything with real consequences.
